In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split

ds = pd.read_csv("goog.csv")
print(ds.head())

# convert the dataframe to a NumPy array
# and remove the time axis
data = ds.values
data = data[:, 1:]
data = np.array(data, dtype=np.float32)

# normalize the data
mean = data.mean(axis=0)
std = data.std(axis=0)
data = (data - mean) / std

# prepare feature-target pairs
# feature: a window of size 'N' i.e. x[i:i+N]
# target: a window of size 'N' right-shifted by 1 i.e. x[i+1:i+1+N]
window_len = 16
x = []
y = []
for i in range(data.shape[0] - window_len):
  x.append(data[i:i+window_len])
  y.append(data[i+1:i+window_len+1])
x = np.array(x)
y = np.array(y)
print(x.shape)
print(y.shape)

# train/test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=42)
print(x_train.shape)
print(x_test.shape)
print(y_train.shape)
print(y_test.shape)

In [ ]:
from keras import layers
from keras import Sequential
from keras import optimizers

model = Sequential([
    layers.Input(shape=(16, 5)),
    layers.RNN(
        layers.StackedRNNCells([
            layers.SimpleRNNCell(32)
            for _ in range(8)
        ]),
        return_sequences=True
    ),
    layers.Dense(5)
])

model.compile(loss="mae", optimizer=optimizers.Adam(learning_rate=0.01), metrics=["mae"])
model.summary()

In [ ]:
model.fit(x_train, y_train, epochs=25, batch_size=4, validation_data=(x_test, y_test))

In [ ]:
import matplotlib.pyplot as plt

seq = ds.values[:,1:]
seq = (seq - mean) / std
seq = seq.tolist()
N = len(seq)

for _ in range(30):
  L = len(seq)
  input_seq = np.array(seq[L-window_len:L])
  input_seq = input_seq.reshape((1, window_len, 5))
  pred = model.predict(input_seq)
  pred = pred.tolist()[0][-1]
  seq.append(pred)

print(N, L)
plt.plot(list(range(len(seq))), [x[3] * std[3] + mean[3] for x in seq])
plt.show()